In [1]:
from openai import OpenAI
import json
from tqdm import tqdm, trange

In [ ]:
client = OpenAI(
    api_key="",
)

In [ ]:
! gdown 

Downloading...
From: https://drive.google.com/uc?id=10yvQXiaHhPEz3iVopohH1TG3mpKI1cr2
To: /kaggle/working/integrated.json
100%|██████████████████████████████████████| 1.55M/1.55M [00:00<00:00, 3.27MB/s]


In [4]:
with open('/kaggle/working/integrated.json', 'r') as f:
    all_data = json.loads(f.read())
    f.close()

In [5]:
def format_message(input_text):
  messages = [
      {
          "role": "system",
          "content": (
                        """You are an expert annotator of diabetes-related stigma in language.
                        "Your task is to classify the presence and type of stigma in a given input text (a quote, part of an interview, or a social media post).

                        Definition of stigma: any form of discrimination, exclusion, negative stereotypes,
                        shame, blame, or harmful beliefs directed at someone because of diabetes.

                        "Classification steps:\n"
                        "1. Determine if the post expresses stigma:\n"
                        "   - stigma: Contains stigmatizing content or implications.\n"
                        "   - no stigma: Neutral, supportive, or stigma-free.\n\n"
                        "2. If 'stigma', identify all applicable types:\n"
                        "   - Experienced: Direct discrimination or exclusion due to diabetes.\n"
                        "   - Perceived: Awareness or assumption that others hold negative views.\n"
                        "   - Anticipated: Fear or expectation of being stigmatized.\n"
                        "   - Internalized: Shame or self-blame from negative diabetes stereotypes.\n"
                        "   - Intersectional: Stigma compounded by other factors (e.g., obesity, race).\n\n"
                        "   If 'no stigma', return an empty list for Type.\n\n"
                        "3. Provide a brief explanation (max 50 words) for your decision.\n\n"
                        "Respond in this exact format:\n"
                        "Label: [Yes stigma / No stigma]\n"
                        "Type: [List of applicable types or []]\n"
                        "Reasoning: [Your explanation]"

                        Examples:
                        Input: "My fiancé’s mother told him not to marry me because I have diabetes."
                        Output:
                        Label: stigma
                        Type: Experienced
                        Reasoning: This statement shows discrimination and exclusion because the person is being judged as a burden due to diabetes.

                        Input: "I check my blood sugar every morning before breakfast."
                        Output:
                        Label: no stigma
                        Reasoning: This is a neutral statement describing a health routine, without stigma or negative judgment.

                        Input: "People say diabetes is only for lazy people who eat too much."
                        Output:
                        Label: stigma
                        Type: Perceived
                        Reasoning: The text conveys a negative stereotype linking diabetes to laziness and overeating.
                        """
                    )
      },
      {
          "role": "user",
          "content": f"Text: {input_text}\n Analyze the text for diabetes-related stigma."
      }
  ]

  return messages


In [6]:
all_responses = []

for data_obj in tqdm(all_data):
    completion = client.chat.completions.create(
        model="gpt-4.1-2025-04-14",
        messages=format_message(data_obj['input text']),
        temperature=0.0,
        max_completion_tokens = 50,
        logprobs=True,
        top_logprobs=10
    )

    response_text = completion.choices[0].message.content
    all_logprobs = []

    for i in range(len(completion.choices[0].logprobs.content)):
      position_tokens = completion.choices[0].logprobs.content[i].top_logprobs
      position_list = []
    
      for i, logprob in enumerate(position_tokens, start=1):
        position_list.append((logprob.token, logprob.logprob))
    
      all_logprobs.append(position_list)

    res_obj = {'answer': response_text, 'all_logprobs': all_logprobs}
    all_responses.append(res_obj)

    if len(all_responses) % 2 == 0:
        with open('gpt4.1_wtype_answers.json', 'w') as f:
            f.write(json.dumps(all_responses))
            f.close()
    

100%|██████████| 1386/1386 [53:23<00:00,  2.31s/it]


In [7]:
with open('final_gpt4.1_wtype_answers.json', 'w') as f:
    f.write(json.dumps(all_responses))
    f.close()